# 15 — Superposition and High-Dimensional Representations

**Description:** Explore how many feature directions can share a representation space, why neurons need not correspond one-to-one with concepts, and how high-dimensional geometry reduces interference.
**Level:** Beginner
**Tags:** Language Models, Superposition, High-Dimensional Geometry, Cosine Similarity, Interpretability

Notebook 14 used one coordinate per entity and attribute. Real models must represent far more potentially useful features than they have individual neurons. **Superposition** is the idea that models can represent many features as directions sharing the same vector space, especially when only a small subset is active at once.

This notebook develops the geometric intuition. It does not claim that random directions fully explain learned superposition.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Neurons are coordinates; features can be directions

In a 2D space, the coordinate axes are only two directions, but we can describe many other directions. A feature vector need not align with either individual neuron.

In [ ]:
angles = np.deg2rad([0, 35, 75, 120, 160])
features_2d = np.stack([np.cos(angles), np.sin(angles)], axis=1)

fig, ax = plt.subplots(figsize=(6, 6))
for i, vector in enumerate(features_2d):
    ax.arrow(0, 0, *vector, width=0.012, length_includes_head=True)
    ax.text(*(1.1 * vector), f"feature {i}", ha="center")
ax.set(xlim=(-1.4, 1.4), ylim=(-1.4, 1.4), aspect="equal", xlabel="neuron 1 coordinate", ylabel="neuron 2 coordinate", title="More feature directions than coordinate axes")
plt.show()

A feature's activation can be measured by projection onto its direction. Because directions overlap, one feature can produce a nonzero response in another detector: **interference**.

## 2. Cosine similarity measures directional overlap

In [ ]:
def cosine_matrix(vectors):
    unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    return unit @ unit.T

similarities_2d = cosine_matrix(features_2d)
print(similarities_2d)
off_diagonal = similarities_2d[~np.eye(len(features_2d), dtype=bool)]
print("largest absolute overlap:", np.abs(off_diagonal).max())

With five directions packed into 2D, substantial overlap is unavoidable. Higher dimensions provide much more room.

## 3. Random directions become nearly orthogonal

Generate unit vectors in several dimensions and measure pairwise cosine similarities.

In [ ]:
rng = np.random.default_rng(15)
dimensions = [2, 8, 32, 128, 512]
number_of_features = 200
distributions = {}

for dimension in dimensions:
    vectors = rng.normal(size=(number_of_features, dimension))
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
    similarities = vectors @ vectors.T
    upper = similarities[np.triu_indices(number_of_features, k=1)]
    distributions[dimension] = upper
    print(f"d={dimension:3d}: mean |cos|={np.mean(np.abs(upper)):.3f}, 95th percentile={np.percentile(np.abs(upper),95):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for dimension in [2, 8, 32, 128]:
    ax.hist(distributions[dimension], bins=60, density=True, histtype="step", linewidth=2, label=f"d={dimension}")
ax.set(xlabel="pairwise cosine similarity", ylabel="density", title="Random directions concentrate near orthogonality in high dimensions")
ax.legend()
plt.show()

This concentration is related to the geometry behind the Johnson–Lindenstrauss lemma: many points can be embedded into a dimension that grows only logarithmically with their count while approximately preserving pairwise distances. Here we use only the intuition, not the theorem as a capacity formula for neural networks.

## 4. Encode sparse features in superposition

Let $F$ feature directions share a $d$-dimensional representation. To encode a sparse feature vector $a$, add its active directions:

$$x=a^TD$$

where rows of $D$ are unit feature directions.

In [ ]:
d = 64
F = 256
directions = rng.normal(size=(F, d))
directions /= np.linalg.norm(directions, axis=1, keepdims=True)

active_ids = np.array([7, 41, 203])
true_activations = np.zeros(F)
true_activations[active_ids] = [1.0, 0.8, 1.2]
representation = true_activations @ directions

decoded_scores = directions @ representation
top_ids = np.argsort(decoded_scores)[-8:][::-1]
print("true active features:", active_ids)
print("top decoded features:", top_ids)
print("top scores:          ", decoded_scores[top_ids])

Projection approximately recovers the active features, but unrelated directions receive small nonzero scores. Those false responses are interference from non-orthogonality.

## 5. Sparsity controls interference

When few features are active at once, fewer directions are summed and decoding is cleaner. We measure whether the strongest decoded scores recover the true active set.

In [ ]:
def recovery_rate(number_active, trials=200):
    recovered = []
    for _ in range(trials):
        active = rng.choice(F, size=number_active, replace=False)
        representation = directions[active].sum(axis=0)
        predicted = np.argpartition(directions @ representation, -number_active)[-number_active:]
        recovered.append(len(set(active) & set(predicted)) / number_active)
    return np.mean(recovered)

active_counts = np.array([1, 2, 4, 8, 16, 32])
rates = np.array([recovery_rate(k) for k in active_counts])

plt.plot(active_counts, rates, "o-")
plt.ylim(0, 1.05)
plt.xlabel("features active simultaneously")
plt.ylabel("mean top-k recovery")
plt.title("Sparse activation makes superposed features easier to recover")
plt.show()
print(dict(zip(active_counts, np.round(rates, 3))))

## 6. Width controls interference

In [ ]:
widths = [16, 32, 64, 128, 256]
mean_overlaps = []
for width in widths:
    D = rng.normal(size=(F, width))
    D /= np.linalg.norm(D, axis=1, keepdims=True)
    pairwise = D @ D.T
    off_diagonal = pairwise[np.triu_indices(F, k=1)]
    mean_overlaps.append(np.mean(np.abs(off_diagonal)))

plt.plot(widths, mean_overlaps, "o-")
plt.xlabel("representation width")
plt.ylabel("mean absolute feature overlap")
plt.title("Wider spaces reduce random directional overlap")
plt.show()

## 7. Why one neuron is not one concept

If features are directions not aligned with coordinate axes:

- one feature influences many neuron coordinates;
- one neuron coordinate participates in many feature directions;
- reading a single neuron's activation mixes contributions from multiple features.

This is **polysemanticity** at the neuron level. Superposition is one proposed reason it appears.

## 8. Important limits of the demonstration

- Random directions are not learned model features.
- Nearly orthogonal does not mean exactly independent.
- Neural networks use nonlinearities, biases, normalization, and multiple layers.
- Feature importance and activation frequency affect which representations are worth learning.
- Johnson–Lindenstrauss gives a geometric preservation result, not a direct count of “concepts inside an LLM.”

## 9. Challenges

1. Repeat the sparse recovery experiment with unequal activation strengths.
2. Add Gaussian noise to the representation and plot recovery.
3. Compare random directions with deliberately orthogonal directions when $F\le d$.
4. Find how large `d` must be before 95% of pairwise absolute cosines are below 0.2 for 500 vectors.
5. Explain why sparse features are especially compatible with superposition.

## Takeaways

- Neurons are coordinate axes; learned features can be arbitrary directions.
- High-dimensional random directions tend to be nearly orthogonal, allowing many low-interference directions.
- Superposition represents more features than dimensions by allowing directions to overlap.
- Sparse activation reduces simultaneous interference and makes recovery easier.
- One feature may use many neurons, and one neuron may participate in many features.
- Notebook 16 introduces sparse autoencoders as one method for discovering a more interpretable feature dictionary.